# 06 — Pipeline Kedro y reproducibilidad (CRISP-DM: fase 6)

Objetivos:
- Ejecutar el proyecto como **flujo versionable**.
- Ubicar **parámetros**, **catálogo** y **artefactos**.

Comandos en terminal (raíz del repo):
```bash
kedro run
kedro run --pipeline data_processing
kedro info
```

In [ ]:
%load_ext kedro.ipython

In [ ]:
from pathlib import Path
import json

# Parámetros cargados por Kedro (mismo contenido que conf/base/parameters.yml)
params = catalog.load("parameters")
print("Split:", params.get("split"))
print("Clasificación — features:", params.get("classification", {}).get("feature_columns"))

> **Nota:** La celda siguiente ejecuta **`session.run()`** (todo el pipeline: datos + clasificación + regresión). Puede tardar varios minutos y requiere `data/raw/database.sqlite`. En clase puedes usar solo la terminal: `kedro run`.


In [ ]:
# Ejecutar todo el pipeline desde el notebook (equivalente a `kedro run` en la raíz)
from kedro.framework.session import KedroSession

root = Path.cwd()
if not (root / "pyproject.toml").is_file():
    raise SystemExit("Abre Jupyter desde la raíz del proyecto Kedro.")

with KedroSession.create(project_path=root) as session:
    session.run()

print("Ejecución completada.")

In [ ]:
# Leer métricas generadas (después de session.run())
import json
from pathlib import Path

paths = {
    "classification": Path("data/08_reporting/classification_metrics.json"),
    "regression": Path("data/08_reporting/regression_metrics.json"),
}
for label, path in paths.items():
    if path.is_file():
        with path.open() as f:
            data = json.load(f)
        print(f"\n=== {label} ===")
        print("best_model:", data.get("best_model"))
        print("leaderboard (muestra):", json.dumps(data.get("leaderboard", [])[:3], indent=2))
    else:
        print(f"Falta {path}; ejecuta la celda `session.run()` o `kedro run` en terminal.")

In [ ]:
# Cargar el Parquet de features y el modelo entrenado (si existen)
import pandas as pd

fp = Path("data/05_model_input/features_for_ml.parquet")
if fp.is_file():
    display(pd.read_parquet(fp).head())
else:
    print("Aún no existe features_for_ml.parquet")